## Optimization With FP8
---

In this notebook, you will learn how to optimize a transformer-based model using the Transformer Engine and FP8 precision. We will investigate each optimization step via the Nsight System GUI. Please note that your results may slightly differ from the stated figures/screenshots in this notebook, depending on your hardware configuration; however, the idea remains the same. Let's get started with a baseline code to benchmark the results from our optimization process.

### Pure PyTorch Baseline

To illustrate FP8 optimization with the Transformer Engine (TE), we will build a basic GPT-style Transformer decoder layer. This will serve as our baseline and will help monitor our optimization progress. The Transformer decoder layer will consist of LayerNorm, QKV Projection, Dot Product Attention, a projection with a residual connection between Dropout and LayerNorm, an MLP, and a final residual connection. Please see the [code details here](../source_code/fp8/basic_gpt.py)

```python
class PyTorchMLP(torch.nn.Module):
    """Feed-forward network in Transformer layer.
    Built with plain PyTorch modules.
    """

    hidden_size: int
    ffn_hidden_size: int
    ...

class PyTorchTransformerLayer(torch.nn.Module):
    """Basic Transformer layer using plain PyTorch modules."""

    def __init__( self,  hidden_size: int, ffn_hidden_size: int, num_attention_heads: int, layernorm_eps: float = 1e-5, attention_dropout: float = 0.1,  hidden_dropout: float = 0.1,  ):
    ...

hidden_size = 4096
sequence_length = 2048
batch_size = 4
ffn_hidden_size = 16384
num_attention_heads = 32
dtype = torch.float16
...    
```

Please run the cell below to capture the baseline execution time.

In [ ]:
!python ../source_code/fp8/basic_gpt.py

**Likely Output on H100**:
```python
Baseline PyTorch:
Mean time: 25.251611328125 ms
```
Now, let's look at the details through profiling. For example, the time it takes to move the model to the device, data to the device, and how long it takes the model to execute. Please run the cell below.

In [ ]:
!cd ../source_code/fp8 && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=all \
--cuda-flush-interval=0 \
--stats=true \
--output ../../reports/transformer_baseline \
--force-overwrite true python basic_gpt.py

**Profiling Stats Screenshot**:
<img src="images/gpt_baseline.png">

### Applying Transformer Engine Unfused Module 

Transformer Engine `Unfused` implies using the TE layers but executing each logical operation (`linear`, `layer norm`, `attention`, etc.) as separate GPU kernels, rather than combining them into a single or a few highly fused kernels. This is implemented by replacing the standard PyTorch modules with TE equivalents. In our sample code, the `torch.nn.Linear` is replaced with `te.Linear` while `torch.nn.LayerNorm` is replaced with `te.LayerNorm`. But each op still runs as its own kernel. This approach requires minimal code changes, better precision handling, and often some speedup vs baseline, but not the maximum possible performance. Please find the [code details here](../source_code/fp8/te_unfused.p). 

```python
class TEUnfusedMLP(torch.nn.Module):
    """MLP using TE modules."""
    ...

class TEUnfusedTransformerLayer(torch.nn.Module):
    """Transformer layer using basic TE modules."""
    ...
    self.ln1 = te.LayerNorm(hidden_size, eps=layernorm_eps)
    self.qkv_projection = te.Linear(hidden_size, 3 * hidden_size, bias=True)
    ...
```
In our code, we build `TEUnfusedMLP` and `TEUnfusedTransformerLayer` classes by swapping these modules (`te.Linear` and `te.LayerNorm`) in, while keeping attention in framework-native form. Let's run the cell below to track the execution time.

In [ ]:
!python ../source_code/fp8/te_unfused.py

**Likely Output on DGXC (H100)**:
```python
TE Unfused:
Mean time: 25.294013671875 ms
```
From the output, the mean execution time is slightly more than the baseline. Let's investigate further by profiling the code to see the stats. Please run the cell below to profile the code.

In [ ]:
!cd ../source_code/fp8 && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=all \
--cuda-flush-interval=0 \
--stats=true \
--output ../../reports/te_unfused \
--force-overwrite true python te_unfused.py

**Profiling Stats Screenshot**:
<img src="images/te_unfused.png"> 
<br/>
**Nsight Systems Screenshot**:
<img src="images/te_unfused_nsys.png"> 
<br>

From the screenshots, we could see that new ops were introduced. This is in line with our code, as we previously introduced `te.Linear` and `te.LayerNorm` modules. These modules' ops correspond to `nvte_cublas_gemm_v2` for matrix multiplication, `nvte_layernorm_bwd` for a layer normalization op during the backward pass, and `nvte_layernorm_fwd` for a layer normalization op during the forward pass. To optimize further, we focus on the dot-product ops, particularly the attention mechanism.


### Using Transformer Engine with Unfused and Attention Modules

The core idea behind Transformer models is the [attention mechanism](https://arxiv.org/abs/1706.03762). It identifies word correlations, selects the most important parts of the sentence to focus on, and captures meaningful patterns and dependencies in the data. The screenshot below shows a typical attention mechanism, where `pre-softmax` operations can be a combination of scaling, bias, and masking, while the `post-softmax` operation is often just dropout. Transformer Engine provides multiple attention backends for each supported framework. The framework-native backends provide a robust baseline, while the fused, GPU-optimized implementations offer more performance. For example, the `flash-attention` and `cuDNN attention` backends in PyTorch. The framework-native backends are often named with `“unfused”`, while the more optimized backends are `“fused”` or `“flash”`.

<img src="images/te_attention.png"> 
<br>

Now let’s replace the attention mechanism with Transformer Engine’s optimized DotProductAttention (`DotProductAttention` → `te.DotProductAttention`) [within our code](../source_code/fp8/te_unfused_attn.py). 

```python

self.attention = te.DotProductAttention(
            num_attention_heads=num_attention_heads,
            kv_channels=self.kv_channels,
            attention_dropout=attention_dropout,
            attn_mask_type="causal",
        )
```
The Transformer Engine’s attention automatically selects the best available backend (`FlashAttention` or `cuDNN fused attention`) based on the hardware and input configuration. This process delivers optimal performance without manual tuning. Please run the cell below to see the execution time. 

In [ ]:
!python ../source_code/fp8/te_unfused_attn.py

**Likely Output on H100:**
```python
TE Unfused + TE Attention:
Mean time: 18.566383056640625 ms
```
The output result shows a reduction in the model execution time due to the introduction of TE attention. Let's visualize this in Nsight Systems and investigate the selected attention backend. Please run the cell below to profile the code.

In [ ]:
!cd ../source_code/fp8 && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=all \
--cuda-flush-interval=0 \
--stats=true \
--output ../../reports/te_unfused_attn \
--force-overwrite true python te_unfused_attn.py

**Profiler Screenshot**
<img src="images/te_unfused_attn_nsys.png"> 
<br>

From the screenshot, it is noticeable that the TE attention automatically selected the `FlashAttention` as the backend and used it for the forward and backward passes (`nvte_flash_attn_bwd`, `nvte_flash_attn_fwd`). The ops resulted in model execution speedup (`2.132 sec.`). Next, we will introduce FP8 precision to further optimize the process.


### Applying FP8 with Transformer Engine Unfused and Attention Modules

In this session, we will introduce FP8 precision in the existing flow. This will be done by wrapping our code within an autocast context manager to enable the precision. This provides significant speedups on Hopper, Ada, and Blackwell GPUs. The code snippet below shows the new additions to our [existing code](../source_code/fp8/te_unfused_attn_fp8.py). Note that the autocast should only wrap the forward pass and must exit before the backward pass starts.

```python
from transformer_engine.common.recipe import Format, DelayedScaling
...

recipe = DelayedScaling( fp8_format=Format.HYBRID, amax_history_len=16, amax_compute_algo="max" )

with te.autocast(enabled=True, recipe=recipe):
    y = te_unfused_fp8(x, attention_mask=None)
...
```

Please run the cell below to see the effect of FP8 enabled. We expect to see more speedups in the model execution time.

In [ ]:
!python ../source_code/fp8/te_unfused_attn_fp8.py

**Likely Output on H100:**
```python
TE Unfused + TE Attention + FP8:
Mean time: 12.3841552734375 ms
```
As expected, we got some speedups. Let's dive deep to examine what actually contributed to the result by profiling our code. Please run the cell below.

In [ ]:
!cd ../source_code/fp8 && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=all \
--cuda-flush-interval=0 \
--stats=true \
--output ../../reports/te_unfused_attn_fp8 \
--force-overwrite true python te_unfused_attn_fp8.py

**Profiling Stats Screenshot**:
<img src="images/te_unfused_attn_fp8.png"> 
<br/>
**Nsight GUI Screenshot**
<img src="images/fp8_stats.png"> 

From the profiling stats screenshot, we can see a reduction in the execution time for `nvte_flash_attn_bwd` and `nvte_flash_attn_fwd` ops. The same applies to the `nvte_layernorm_fwd` and `nvte_layernorm_bwd` processes. Besides this, we notice traces of `nvte_delayed_scaling_recipe_amax_and_scale_update_after_reduction`, `E4M3`, and `E5M2`, which correspond to FP8 Hybrid format enabled.

### Using Optimized Modules (TE Fused, TE Attention module, and FP8)

Fused modules use kernel fusion to combine multiple operations. It combines several steps (e.g., LayerNorm + Linear, or the entire scaled dot-product attention) into one kernel, minimizing memory writes and launches. While speedups are modest on a single GPU, they scale better in multi-GPU setups. Combined with TE Attention and FP8, this delivers peak performance by reducing memory bandwidth usage, minimizing kernel-launch overhead, improving cache utilization, and eliminating intermediate memory allocations. Available Fused modules include:

- `te.LayerNormLinear` - fuses LayerNorm + Linear
- `te.LayerNormMLP` - fuses LayerNorm + MLP

```python
...
# Fused LayerNorm + QKV projection
self.ln_qkv = te.LayerNormLinear(hidden_size, 3 * hidden_size, eps=layernorm_eps, bias=True)

...
# Fused LayerNorm + MLP
self.ln_mlp = te.LayerNormMLP(hidden_size, ffn_hidden_size, eps=layernorm_eps, bias=True)

...
```
Please find the detailed [implementation code here](../source_code/fp8/te_optimized_modules). Let's run the cell below to track the execution time.

In [ ]:
!python ../source_code/fp8/te_optimized_modules.py

**Likely Output on H100:**
```python
TE Fused + TE Attention + FP8:
Mean time: 12.19614501953125 ms
```

In [ ]:
!cd ../source_code/fp8 && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=all \
--cuda-flush-interval=0 \
--stats=true \
--output ../../reports/te_optimized_modules \
--force-overwrite true python te_optimized_modules.py

**Nsight GUI Screenshot**

<img src="images/te_optimized_module_nsys.png"> 
<br>

From the screenshot, we can see a further reduction in the execution time for `nvte_flash_attn_bwd`, `nvte_flash_attn_fwd`, `nvte_layernorm_fwd` and `nvte_layernorm_bwd` ops. This resulted in further speedups of our code. 

### Using TE TransformerLayer Module and FP8

Transformer Engine provides a ready-to-use `TransformerLayer` module that includes all optimizations out of the box. To implement this, use the `te.TransformerLayer`. The `TransformerLayer` provides the simplest integration as all processes are handled automatically. The code snippet below shows how to implement the `TransformerLayer`

```python
te_transformer_layer = (
    te.TransformerLayer(
        hidden_size=hidden_size,
        ffn_hidden_size=ffn_hidden_size,
        num_attention_heads=num_attention_heads,
        self_attn_mask_type="causal",
        layernorm_epsilon=1e-5,
        bias=True,
        hidden_dropout=0.0,
        attention_dropout=0.0,
    )
    .to(dtype=dtype)
    .cuda()
)
```

Please find the detailed code with FP8 implementation [here](../source_code/fp8/te_transformer_layer_fp8.py). Let's run the cell below to see the output of our final optimization steps.


In [ ]:
!python ../source_code/fp8/te_transformer_layer_fp8.py

**Likely Output on H100:**
```python
TE TransformerLayer + FP8:
Mean time: 11.266163330078125 ms
```
The `TransformerLayer` gave further speedups reducing the execution time from `12.19 ms` to `11.26 ms`.


In [ ]:
!cd ../source_code/fp8 && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=all \
--cuda-flush-interval=0 \
--stats=true \
--output ../../reports/te_transformer_layer_fp8 \
--force-overwrite true python te_transformer_layer_fp8.py

### Summary

The table below summarizes the performance improvements achieved with Transformer Engine on an NVIDIA H100 GPU. Results may vary depending on hardware and configuration. While this tutorial focuses on a simple single-GPU scenario, features like fused layers can provide additional benefits in more complex setups such as multi-GPU training.


|Implementation modules  | Time (ms)  | Speedup   |
| --| -- | -- |
| Baseline PyTorch                |  25.25  | 1.00x  |
|TE Unfused                       |  25.29  | 0.99x  |
| TE Unfused + TE Attention       |  18.56  | 1.36x  |
|TE Unfused + TE Attention + FP8  |  12.38  | 2.03x  |
|TE Fused + TE Attention + FP8    |  12.19  | 2.07x  |
|TE TransformerLayer + FP8        |  11.26  | 2.24x  | 

---
### References
- https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/getting_started/index.html
- https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/examples/attention/attention.html
- https://puzzles.modular.com/puzzle_22/forward_pass.html


## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.